<a href="https://colab.research.google.com/github/andysanchez1403-droid/SIMULACION-II/blob/main/Caminata_Aleatoria_sobre_un_grafo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Caminata Aleatoria sobre un Grafo: Estimación del Número Esperado de $1$'s en una Buena Cadena**

## 1. Formulación del Problema

Consideremos el espacio de secuencias binarias de longitud $m$, $X = (x_1, x_2, \dots, x_m) \in \{0, 1\}^m$. Decimos que una secuencia binaria es **buena** (o admisible) si **no contiene unos ($1$'s) adyacentes**:
$$x_i \cdot x_{i+1} = 0, \quad \forall \, i \in \{1, 2, \dots, m-1\}$$


# **Probabilidad**
Para una caminata aleatoria de longitud $N$ representada por la secuencia de variables aleatorias independientes e idénticamente distribuidas (i.i.d.) $X_1, X_2, \dots, X_N \sim \text{Bernoulli}(p = 0.5)$, consideramos el espacio muestral de todas las sub-tuplas de longitud $m = 4$ extraídas mediante una \textbf{ventana deslizante}.

El número total de sub-tuplas de $m = 4$ bits analizadas en una cadena de longitud $N$ es:
\[
T_{\text{totales}} = N - m + 1 = N - 3
\]

Dado que existen $2^m = 2^4 = 16$ combinaciones posibles en total, y de las cuales exactamente $F_6 = 8$ son secuencias buenas (\textbf{0000, 0001, 0010, 0100, 1000, 0101, 1010, 1001}), la probabilidad teórica de que una sub-tupla arbitraria sea buena bajo un proceso estocástico no restringido es:
$[
P(\text{Tupla Buena}) = \frac{N(4)}{2^4} = \frac{8}{16} = 0.5000 \quad (50.00\%)
]$

Usando la librería $\texttt{random}$, empezamos a generar la tupla buena:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética de gráficas
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

In [2]:
def is_good_sequence(seq: tuple) -> bool:
    """
    Verifica si una secuencia binaria no contiene unos (1s) adyacentes.
    """
    for i in range(len(seq) - 1):
        if seq[i] == 1 and seq[i + 1] == 1:
            return False
    return True

def get_good_sequences(m: int) -> list:
    """
    Genera exhaustivamente todas las buenas secuencias binarias de longitud m.
    El número total de secuencias cumple |Omega_m| = F_{m+2}
    """
    good_seqs = []
    total_possibilities = 1 << m  # 2^m posibles combinaciones

    for i in range(total_possibilities):
        seq = tuple((i >> (m - 1 - j)) & 1 for j in range(m))
        if is_good_sequence(seq):
            good_seqs.append(seq)

    return good_seqs

# Ejemplo práctico para m = 4
m = 4
good_seqs = get_good_sequences(m)
print(f"Número total de buenas cadenas para m = {m}: {len(good_seqs)}")
print("Secuencias admisibles:")
for seq in good_seqs:
    print("  ", "".join(map(str, seq)))

Número total de buenas cadenas para m = 4: 8
Secuencias admisibles:
   0000
   0001
   0010
   0100
   0101
   1000
   1001
   1010


In [3]:
def build_transition_matrix(m: int):
    """
    Construye la matriz de transición P para las buenas secuencias de longitud m.
    Implementa una caminata aleatoria perezosa (La buena caminata sobre un grafo) con rechazo.
    """
    good_seqs = get_good_sequences(m)
    n = len(good_seqs)
    seq_to_idx = {seq: i for i, seq in enumerate(good_seqs)}

    P = np.zeros((n, n))

    for i, seq in enumerate(good_seqs):
        for bit_idx in range(m):
            seq_list = list(seq)
            seq_list[bit_idx] = 1 - seq_list[bit_idx]
            candidate_seq = tuple(seq_list)

            if is_good_sequence(candidate_seq):
                j = seq_to_idx[candidate_seq]
                P[i, j] += 1.0 / m
            else:
                # Transición rechazada
                P[i, i] += 1.0 / m

    return good_seqs, P, seq_to_idx

good_seqs, P, seq_to_idx = build_transition_matrix(m=4)

print("Matriz de transición P (m=4, redondeada a 3 decimales):")
print(np.round(P, 3))

Matriz de transición P (m=4, redondeada a 3 decimales):
[[0.   0.25 0.25 0.25 0.   0.25 0.   0.  ]
 [0.25 0.25 0.   0.   0.25 0.   0.25 0.  ]
 [0.25 0.   0.5  0.   0.   0.   0.   0.25]
 [0.25 0.   0.   0.5  0.25 0.   0.   0.  ]
 [0.   0.25 0.   0.25 0.5  0.   0.   0.  ]
 [0.25 0.   0.   0.   0.   0.25 0.25 0.25]
 [0.   0.25 0.   0.   0.   0.25 0.5  0.  ]
 [0.   0.   0.25 0.   0.   0.25 0.   0.5 ]]


In [4]:
def compute_theoretical_pi(P: np.ndarray) -> np.ndarray:
    """
    Resuelve el sistema pi * P = pi sujeto a sum(pi) = 1 mediante álgebra lineal.
    """
    n = P.shape[0]
    A = (P.T - np.eye(n))
    A[-1, :] = 1.0  # Sustituir la última fila por la restricción de normalización
    b = np.zeros(n)
    b[-1] = 1.0

    pi = np.linalg.solve(A, b)
    return pi

pi_teorico = compute_theoretical_pi(P)
num_ones = np.array([sum(seq) for seq in good_seqs])

# Esperanza teórica E[r(X)] = sum( r(j) * pi_j )
esperanza_teorica = np.dot(num_ones, pi_teorico)

print("Distribución Estacionaria Teórica pi:")
for seq, p in zip(good_seqs, pi_teorico):
    print(f"  pi({''.join(map(str, seq))}) = {p:.4f}")

print(f"\nNúmero esperado teórico de 1's para m={m}: E[r(X)] = {esperanza_teorica:.4f}")

Distribución Estacionaria Teórica pi:
  pi(0000) = 0.1250
  pi(0001) = 0.1250
  pi(0010) = 0.1250
  pi(0100) = 0.1250
  pi(0101) = 0.1250
  pi(1000) = 0.1250
  pi(1001) = 0.1250
  pi(1010) = 0.1250

Número esperado teórico de 1's para m=4: E[r(X)] = 1.2500


In [6]:
import random

def buena(tupla_4bits):
    for i in range(len(tupla_4bits) - 1):
        if tupla_4bits[i] == 1 and tupla_4bits[i + 1] == 1:
            return False
    return True

# **2. Construcción de la Caminata y Métricas Empíricas**

Definimos el proceso de simulación para generar una secuencia estocástica de **N** caracteres y evaluar mediante la función buena() la proporción de ventanas contiguas de tamaño **m=4** que satisfacen la condición de no contigüidad de unos:

In [7]:
def probabilidad_buenas(N_caracteres):
    # 1. Generar la caminata aleatoria de N caracteres
    cadena = [random.choice([0, 1]) for _ in range(N_caracteres)]

    # 2. Extraer todas las tuplas de tamaño 4
    total_tuplas_evaluadas = N_caracteres - 3
    tuplas_buenas = 0

    for i in range(total_tuplas_evaluadas):
        sub_tupla = tuple(cadena[i : i + 4])
        if buena(sub_tupla):
            tuplas_buenas += 1

    # 3. Calcular la probabilidad
    probabilidad = tuplas_buenas / total_tuplas_evaluadas
    porcentaje = probabilidad * 100

    return tuplas_buenas, total_tuplas_evaluadas, probabilidad, porcentaje

Se ejecuta la simulación con una longitud de trayectoria de **N=10,000** pasos para verificar la convergencia de la frecuencia relativa hacia la probabilidad teórica **P(Buena)= 0.50**
.

In [8]:
N = 10000  # Longitud N de la caminata
buenas, totales, prob, pct = probabilidad_buenas(N)


print(f"Total de tuplas analizadas: {totales}")
print(f"Tuplas 'buenas' encontradas: {buenas}")
print(f"Probabilidad de que sea buena: {prob:.4f}")
print(f"Porcentaje de tuplas buenas: {pct:.2f}%")

Total de tuplas analizadas: 9997
Tuplas 'buenas' encontradas: 4949
Probabilidad de que sea buena: 0.4950
Porcentaje de tuplas buenas: 49.50%
